In [1]:
import os
import glob 
import re 

import numpy as np 
import pandas as pd 

In [2]:
def df_to_latex(df, decimals = 1, include_header = True):

    lines = []
    print(df.index)

    if include_header:
        lines.append(' & '.join(str(c) for c in df.columns) + ' \\\\')
        
    for row in df.values:
        formatted = ' & '.join(f'{v:.{decimals}f}' for v in row)
        lines.append(formatted + ' \\\\')

    print('\n'.join(lines))


def preprocess(data_path, metrics): 
    
    df = pd.read_csv(data_path)
    df[['id', 'exp']] = df['experiment'].str.split('_', n = 1, expand = True)

    # convert to the proper units
    metrics_to_multiply = ['avg_jaccard', 'delta_x_avg', 'occlusion_acc']
    df[metrics_to_multiply] = (df[metrics_to_multiply] * 100)
    mask = (df['dataset'] == 'cmupanoptic_3dgs') | (df['dataset'] == 'dex_ycb')
    alt_mask = (df['dataset'] == 'kubric-multiview')
    df.loc[mask, 'mte'] = df.loc[mask, 'mte'] * 100
    df.loc[alt_mask, 'mte'] = df.loc[alt_mask, 'mte'] * 13

    # pivot the df to match the format in the paper
    df_pivot = df.pivot(index='experiment', columns='dataset', values=metrics)
    df_pivot.columns = [f'{metric}_{dataset}' for metric, dataset in df_pivot.columns]

    return df_pivot

def preprocess_cotracker(data_path, metrics):
    df = pd.read_csv(data_path)
    df.columns = [c.replace('cotracker_', '') for c in df.columns]

    metrics_to_multiply = ['avg_jaccard', 'delta_x_avg', 'occlusion_acc']
    df[metrics_to_multiply] = df[metrics_to_multiply] * 100
    mask = (df['dataset'] == 'cmupanoptic_3dgs') | (df['dataset'] == 'dex_ycb')
    alt_mask = (df['dataset'] == 'kubric-multiview')
    df.loc[mask, 'mte'] = df.loc[mask, 'mte'] * 100
    df.loc[alt_mask, 'mte'] = df.loc[alt_mask, 'mte'] * 13

    df = df.set_index('dataset')[metrics]

    # flatten to one row: metric_dataset columns
    result = {
        f'{metric}_{dataset}': df.loc[dataset, metric]
        for dataset in df.index
        for metric in metrics
    }
    return pd.DataFrame([result], index=['cotracker'])

# generate summary for the ncam ablations
def combine_ncam_summaries(prefix, summary_name='summary_metrics.csv', 
                           out_name='summary.csv'):

    dfs = []

    for path in sorted(glob.glob(os.path.join(prefix, 'nviews-*', '*', summary_name))):
        n_views = int(re.search(r'nviews-(\d+)', path).group(1))
        df = pd.read_csv(path)
        df.insert(0, 'n_views', n_views)
        dfs.append(df)

    combined = pd.concat(dfs, ignore_index=True).sort_values('n_views').reset_index(drop=True)
    out_path = os.path.join(prefix, out_name)
    combined.to_csv(out_path, index=False)
    print(f'Saved {len(combined)} rows to {out_path}')

    return combined

import json

def load_results_df(root, agg='mean'):
    """Scan model run dirs under root and return a wide-format summary DataFrame.

    Each subdir must contain a summary.json written by eval_testset_metrics.py.
    Returns one row per experiment, with columns named {metric}_{dataset}.
    Datasets with 0 trials are included with NaN values.

    Args:
        root: path like '/groups/karashchuk/home/ruppk2/results/paper/v1'
        agg:  'mean' or 'median'
    """
    suffix = f'_{agg}'
    rows = {}
    for exp_dir in sorted(os.listdir(root)):
        summary_path = os.path.join(root, exp_dir, 'summary.json')
        if not os.path.exists(summary_path):
            continue
        with open(summary_path) as f:
            summary = json.load(f)
        row = {}
        for dataset, stats in summary.items():
            if not dataset:
                continue
            for key, val in stats.items():
                if key.endswith(suffix):
                    metric = key[: -len(suffix)]
                    row[f'{metric}_{dataset}'] = val
        rows[exp_dir] = row
    df = pd.DataFrame.from_dict(rows, orient='index')
    df.index.name = 'experiment'
    
    return df

def preprocess_posetail(prefix, metrics, datasets):

    df = load_results_df(prefix, agg = 'mean')
    df.to_csv(os.path.join(prefix, 'summary.csv'))
    print(datasets)
    
    for metric in metrics:
        for dataset in datasets:
            col = f'{metric}_{dataset}'
            if col not in df.columns:
                continue

            if metric == 'mte':
                scale = 13 if 'kubric' in dataset else 100 if dataset in ('dex_ycb', 'cmupanoptic_3dgs') else 1
            else:                       # avg_jaccard, delta_x_avg, occlusion_acc -> percent
                scale = 100

            df[col] *= scale

    return df 


def preprocess_mvtracker(run_path, metrics, datasets):

    dataset_map = {
        'dex-ycb-multiview': 'dex_ycb',
        'panoptic-multiview': 'cmupanoptic_3dgs',
        'kubric-multiview': 'kubric-multiview',
    }
    
    with open(os.path.join(run_path, 'summary.json')) as f:
        summary = json.load(f)

    row = {}
    for raw_dataset, stats in summary.items():
        dataset = dataset_map.get(raw_dataset, raw_dataset)
        if dataset not in datasets:
            continue
        for metric in metrics:
            key = f'{metric}_mean'
            if key in stats:
                row[f'{metric}_{dataset}'] = stats[key]

    df = pd.DataFrame([row])

    for metric in metrics:
        for dataset in datasets:
            col = f'{metric}_{dataset}'
            if col not in df.columns:
                continue

            if metric == 'mte':
                scale = 13 if 'kubric' in dataset else 100 if dataset in ('dex_ycb', 'cmupanoptic_3dgs') else 1
            else:                       # avg_jaccard, delta_x_avg, occlusion_acc -> percent
                scale = 100
                
            df[col] *= scale

    return df


def preprocess_mvtracker_csv(run_path, metrics, datasets):
    df = pd.read_csv(os.path.join(run_path, 'combined_summary_metrics.csv'))
    df.columns = [c.replace('mvtracker_', '') for c in df.columns]
    df = df[df['dataset'].isin(datasets)].set_index('dataset')

    row = {}
    for dataset in df.index:
        for metric in metrics:
            if metric not in df.columns:
                continue
            if metric == 'mte':
                scale = 13 if 'kubric' in dataset else 100 if dataset in ('dex_ycb', 'cmupanoptic_3dgs') else 1
            else:                       # avg_jaccard, delta_x_avg, occlusion_acc -> percent
                scale = 100
            row[f'{metric}_{dataset}'] = df.loc[dataset, metric] * scale

    return pd.DataFrame([row])



### data loading and preprocessing

In [9]:
# LOAD 4 CAM DATA 
# load cotracker data 
PREFIX = '/groups/karashchuk/home/ruppk2/results/paper-final-v2/'
cotracker_summary_path = os.path.join(PREFIX, 'cotracker-outputs/paper-canonical-cached/combined_summary_metrics.csv')
cotracker_summary_path_po = os.path.join(PREFIX, 'cotracker-outputs/paper-po-4cams/combined_summary_metrics.csv')
cotracker_df = pd.read_csv(cotracker_summary_path)
cotracker_df_po = pd.read_csv(cotracker_summary_path_po)
metrics = ['avg_jaccard', 'delta_x_avg', 'occlusion_acc', 'mte']
cotracker_df = preprocess_cotracker(cotracker_summary_path, metrics)
cotracker_df_po = preprocess_cotracker(cotracker_summary_path_po, metrics)

# load mvtracker data (4 cams duster + gt)
datasets = ['cmupanoptic_3dgs', 'dex_ycb', 'kubric-multiview']
datasets_po = ['po-animal-sub256-motion', 'point-odyssey-human']

mvtracker_path_duster = os.path.join(PREFIX, 'mvtracker-outputs/paper-canonical-cached-duster')
mvtracker_path_duster_po = os.path.join(PREFIX, 'mvtracker-outputs/paper-po-4cams-duster_0.5')
mvtracker_df_duster = preprocess_mvtracker_csv(mvtracker_path_duster, metrics, datasets)
mvtracker_df_duster_po = preprocess_mvtracker_csv(mvtracker_path_duster_po, metrics, datasets_po)
mvtracker_df_duster = pd.concat([mvtracker_df_duster, mvtracker_df_duster_po], axis=1)

mvtracker_path_gt = os.path.join(PREFIX, 'mvtracker-outputs/paper-canonical-cached-gt')
mvtracker_path_gt_po = os.path.join(PREFIX, 'mvtracker-outputs/paper-po-4cams-gt_0.5')
mvtracker_df_gt = preprocess_mvtracker_csv(mvtracker_path_gt, metrics, datasets)
mvtracker_df_gt_po = preprocess_mvtracker_csv(mvtracker_path_gt_po, metrics, datasets_po)
mvtracker_df_gt = pd.concat([mvtracker_df_gt, mvtracker_df_gt_po], axis=1)

# load posetail data
# datasets = ['cmupanoptic_3dgs', 'dex_ycb', 'kubric-multiview', 
#             'kubric-multiview-moving', 'po-animal-moving-sub256-motion',
#             'po-animal-sub256-motion', 'po-human-moving-small', 
#             'point-odyssey-human', 'tapvid2d', 'tapvid3d_adt', 
#             'tapvid3d_drivetrack', 'tapvid3d_pstudio']
datasets = ['cmupanoptic_3dgs', 'dex_ycb', 'kubric-multiview', 
            'po-animal-sub256-motion', 'point-odyssey-human']
metrics = ['avg_jaccard', 'delta_x_avg', 'occlusion_acc', 'mte']
main_prefix = '/groups/karashchuk/home/ruppk2/results/paper-final-v2/posetail-outputs/'
df = preprocess_posetail(main_prefix, metrics, datasets)

['cmupanoptic_3dgs', 'dex_ycb', 'kubric-multiview', 'po-animal-sub256-motion', 'point-odyssey-human']


In [10]:
# LOAD ALL CAM DATA 
# load cotracker data
datasets_po = ['po-animal-sub256-motion', 'point-odyssey-human']
metrics = ['avg_jaccard', 'delta_x_avg', 'occlusion_acc', 'mte']
cols = [f'{metric}_{dataset}' for dataset in datasets_po for metric in metrics]
 
 # load cotracker_data
cotracker_summary_path = os.path.join(PREFIX, 'cotracker-outputs/paper-po-9cams/combined_summary_metrics.csv')
cotracker_df_all = pd.read_csv(cotracker_summary_path)
metrics = ['avg_jaccard', 'delta_x_avg', 'occlusion_acc', 'mte']
cotracker_df_all = preprocess_cotracker(cotracker_summary_path, metrics)

# load mvtracker data (all cams duster + gt)
mvtracker_path_duster_po = os.path.join(PREFIX, 'mvtracker-outputs/paper-po-9cams-duster_0.5')
mvtracker_df_duster_all = preprocess_mvtracker_csv(mvtracker_path_duster_po, metrics, datasets_po)

mvtracker_path_gt_po = os.path.join(PREFIX, 'mvtracker-outputs/paper-po-9cams-gt_0.5')
mvtracker_df_gt_all = preprocess_mvtracker_csv(mvtracker_path_gt_po, metrics, datasets_po)

# load posetail data
main_prefix = os.path.join(PREFIX, 'posetail-outputs')
df_all = preprocess_posetail(main_prefix, metrics, datasets)
df_all = df_all.loc[['main_finetune_vzkjqrys_po_9cams'], cols]


['cmupanoptic_3dgs', 'dex_ycb', 'kubric-multiview', 'po-animal-sub256-motion', 'point-odyssey-human']


In [11]:
# ncam ablation data 

# cotracker 
cotracker_ncam_path = '/groups/karashchuk/home/ruppk2/results/paper-final-v2/cotracker-outputs/ncam-ablation-dexycb/ncam_ablation_summary.csv'
df_ncam_cotracker = pd.read_csv(cotracker_ncam_path)
avg_jaccard_cotracker = np.round(df_ncam_cotracker['cotracker_avg_jaccard'] * 100, 1)
avg_jaccard_cotracker.index = df_ncam_cotracker['n_views']
print(list(avg_jaccard_cotracker))

# mvtracker 
mvtracker_ncam_path = '/groups/karashchuk/home/ruppk2/results/paper-final-v2/mvtracker-outputs/ncam-ablation-dexycb/ncam_ablation_summary.csv'
df_ncam_mvtracker = pd.read_csv(mvtracker_ncam_path)
avg_jaccard_mvtracker = np.round(df_ncam_mvtracker['mvtracker_avg_jaccard'] * 100, 1)
avg_jaccard_mvtracker.index = df_ncam_mvtracker['n_views']
print(list(avg_jaccard_mvtracker))

# posetail
posetail_ncam_path = '/groups/karashchuk/home/ruppk2/results/paper-final-v2/posetail-outputs/ncam_ablation_dexycb/ncam_ablation_summary.csv'
df_ncam_posetail = pd.read_csv(posetail_ncam_path)
avg_jaccard_posetail = np.round(df_ncam_posetail['avg_jaccard'] * 100, 1)
avg_jaccard_posetail.index = df_ncam_posetail['n_views']
print(list(avg_jaccard_posetail))

# save to df for figure
df_ncam = pd.concat((avg_jaccard_cotracker, avg_jaccard_mvtracker, avg_jaccard_posetail), axis = 1, sort = True)
df_ncam.columns = ['CoTracker3 Baseline', 'MVTracker Baseline', 'Tracktail']
df_ncam.index.name = 'n_views'
df_ncam.to_csv('/home/ruppk2@hhmi.org/tracktail-paper/data/v2/dexycb_ncam_results.csv')

[60.6, 63.7, 62.8, 62.6, 64.1, 65.5, 66.4]
[62.0, 64.0, 68.8, 71.5, 73.0, 73.8, 77.9, 78.0]
[59.9, 74.1, 80.1, 82.1, 82.9, 84.2, 85.5, 85.7]


### main table (table 1)

In [12]:
# main table
experiments = ['main_finetune_vzkjqrys_canonical_cached']
datasets = ['cmupanoptic_3dgs', 'dex_ycb', 'kubric-multiview']
metrics = ['avg_jaccard', 'delta_x_avg', 'occlusion_acc', 'mte']
cols = [f'{metric}_{dataset}' for dataset in datasets for metric in metrics]

# cotracker baseline main metrics (4 cams)
print('cotracker')
sub_df_cotracker = cotracker_df[cols]
df_to_latex(sub_df_cotracker, decimals = 1, include_header = True)
print('')

# mvtracker (4 cams duster)
print('mvtracker, duster')
sub_df_mvtracker_duster = mvtracker_df_duster[cols]
df_to_latex(sub_df_mvtracker_duster, decimals = 1, include_header = True)
print('')

# mvtracker (4 cams gt)
print('mvtracker, gt')
datasets_mvtracker_gt = ['cmupanoptic_3dgs', 'kubric-multiview']
cols_mvtracker_gt = [f'{metric}_{dataset}' for dataset in datasets_mvtracker_gt for metric in metrics]
sub_df_mvtracker_gt = mvtracker_df_gt[cols_mvtracker_gt]
df_to_latex(sub_df_mvtracker_gt, decimals = 1, include_header = True)
print('')

# posetail (4 cams)
print('tracktail')
sub_df_posetail = df.loc[experiments, cols]
df_to_latex(sub_df_posetail, decimals = 1, include_header = True)

cotracker
Index(['cotracker'], dtype='object')
avg_jaccard_cmupanoptic_3dgs & delta_x_avg_cmupanoptic_3dgs & occlusion_acc_cmupanoptic_3dgs & mte_cmupanoptic_3dgs & avg_jaccard_dex_ycb & delta_x_avg_dex_ycb & occlusion_acc_dex_ycb & mte_dex_ycb & avg_jaccard_kubric-multiview & delta_x_avg_kubric-multiview & occlusion_acc_kubric-multiview & mte_kubric-multiview \\
62.5 & 70.3 & 81.5 & 7.6 & 66.4 & 72.8 & 84.4 & 3.7 & 42.9 & 48.8 & 74.5 & 7.8 \\

mvtracker, duster
RangeIndex(start=0, stop=1, step=1)
avg_jaccard_cmupanoptic_3dgs & delta_x_avg_cmupanoptic_3dgs & occlusion_acc_cmupanoptic_3dgs & mte_cmupanoptic_3dgs & avg_jaccard_dex_ycb & delta_x_avg_dex_ycb & occlusion_acc_dex_ycb & mte_dex_ycb & avg_jaccard_kubric-multiview & delta_x_avg_kubric-multiview & occlusion_acc_kubric-multiview & mte_kubric-multiview \\
58.8 & 70.0 & 91.2 & 10.2 & 73.4 & 82.3 & 91.3 & 2.0 & 51.5 & 67.2 & 79.0 & 3.7 \\

mvtracker, gt
RangeIndex(start=0, stop=1, step=1)
avg_jaccard_cmupanoptic_3dgs & delta_x_avg_c

### point odyssey

In [13]:
# PO benchmark 
experiments = ['main_finetune_vzkjqrys_po_4cams']
experiments_all = ['main_finetune_vzkjqrys_po_9cams']
# datasets = ['point-odyssey-human', 'po-animal-sub256-motion',
#             'po-animal-moving-sub256-motion','po-human-moving-small']
datasets = ['point-odyssey-human', 'po-animal-sub256-motion']
metrics = ['avg_jaccard', 'delta_x_avg', 'occlusion_acc', 'mte']
cols = [f'{metric}_{dataset}' for dataset in datasets for metric in metrics]


# COTRACKER (4 cams, 8 cams)
print('cotracker')
df_to_latex(cotracker_df_po[cols], decimals = 1, include_header = True)
df_to_latex(cotracker_df_all[cols], decimals = 1, include_header = True)
print('')

# MVTRACKER duster depth (4 cams, 8 cams)
print('mvtracker, duster')
df_to_latex(mvtracker_df_duster[cols], decimals = 1, include_header = True)
df_to_latex(mvtracker_df_duster_all[cols], decimals = 1, include_header = True)
print('')

# MVTRACKER gt depth (4 cams, 8 cams)
print('mvtracker, gt')
df_to_latex(mvtracker_df_gt[cols], decimals = 1, include_header = True)
df_to_latex(mvtracker_df_gt_all[cols], decimals = 1, include_header = True)

print('tracktail')
df_4 = df.loc[experiments, cols]
df_9 = df.loc[experiments_all, cols]
df_to_latex(df_4, decimals = 1, include_header = True)
df_to_latex(df_9, decimals = 1, include_header = True)
print('') 


cotracker
Index(['cotracker'], dtype='object')
avg_jaccard_point-odyssey-human & delta_x_avg_point-odyssey-human & occlusion_acc_point-odyssey-human & mte_point-odyssey-human & avg_jaccard_po-animal-sub256-motion & delta_x_avg_po-animal-sub256-motion & occlusion_acc_po-animal-sub256-motion & mte_po-animal-sub256-motion \\
22.4 & 24.4 & 52.8 & 4.5 & 25.4 & 28.7 & 52.0 & 2.1 \\
Index(['cotracker'], dtype='object')
avg_jaccard_point-odyssey-human & delta_x_avg_point-odyssey-human & occlusion_acc_point-odyssey-human & mte_point-odyssey-human & avg_jaccard_po-animal-sub256-motion & delta_x_avg_po-animal-sub256-motion & occlusion_acc_po-animal-sub256-motion & mte_po-animal-sub256-motion \\
24.5 & 27.0 & 71.0 & 10.6 & 27.8 & 33.1 & 70.0 & 3.2 \\

mvtracker, duster
RangeIndex(start=0, stop=1, step=1)
avg_jaccard_point-odyssey-human & delta_x_avg_point-odyssey-human & occlusion_acc_point-odyssey-human & mte_point-odyssey-human & avg_jaccard_po-animal-sub256-motion & delta_x_avg_po-animal-sub256

### ablation studies

In [57]:
# cam self attention ablation
experiments = ['ablation_cam_self_attn_folhop3e_canonical_cached', 'main_pretrain_e25w9v86_canonical_cached']
datasets = ['kubric-multiview']
metrics = ['avg_jaccard', 'delta_x_avg', 'occlusion_acc', 'mte']

cols = [f'{metric}_{dataset}' for dataset in datasets for metric in metrics]
sub_df = df.loc[experiments, cols]
df_to_latex(sub_df, decimals = 1, include_header = True)
sub_df

Index(['ablation_cam_self_attn_folhop3e_canonical_cached', 'main_pretrain_e25w9v86_canonical_cached'], dtype='object', name='experiment')
avg_jaccard_kubric-multiview & delta_x_avg_kubric-multiview & occlusion_acc_kubric-multiview & mte_kubric-multiview \\
48.5 & 60.0 & 80.7 & 5.0 \\
59.0 & 70.5 & 86.0 & 2.7 \\


,avg_jaccard_kubric-multiview,delta_x_avg_kubric-multiview,occlusion_acc_kubric-multiview,mte_kubric-multiview
experiment,,,,
ablation_cam_self_attn_folhop3e_canonical_cached,48.546284,59.956524,80.728440,5.01664
main_pretrain_e25w9v86_canonical_cached,58.975121,70.549932,85.993011,2.73745


In [49]:
# # dataset ablation 
# # experiments = ['l60u31gr_ablation_dataset_kubric_only', 'r2eufrxl_ablation_dataset_static_only', '2p9of8sk_main_pretrain']
# experiments = ['ablation_dataset_static_kub_only_f4wpvece_canonical_cached', 'main_pretrain_e25w9v86_canonical_cached']
# datasets = ['cmupanoptic_3dgs', 'dex_ycb', 'kubric-multiview']
# metrics = ['avg_jaccard', 'delta_x_avg', 'occlusion_acc', 'mte']

# cols = [f'{metric}_{dataset}' for dataset in datasets for metric in metrics]
# sub_df = df.loc[experiments, cols]
# df_to_latex(sub_df, decimals = 1, include_header = True)
# sub_df

Index(['ablation_dataset_static_kub_only_f4wpvece_canonical_cached', 'main_pretrain_e25w9v86_canonical_cached'], dtype='object', name='experiment')
avg_jaccard_cmupanoptic_3dgs & delta_x_avg_cmupanoptic_3dgs & occlusion_acc_cmupanoptic_3dgs & mte_cmupanoptic_3dgs & avg_jaccard_dex_ycb & delta_x_avg_dex_ycb & occlusion_acc_dex_ycb & mte_dex_ycb & avg_jaccard_kubric-multiview & delta_x_avg_kubric-multiview & occlusion_acc_kubric-multiview & mte_kubric-multiview \\
78.0 & 91.2 & 88.7 & 4.1 & 82.0 & 90.3 & 92.7 & 0.8 & 64.1 & 75.0 & 87.7 & 2.1 \\
75.6 & 91.7 & 86.1 & 3.9 & 80.8 & 90.7 & 91.1 & 0.7 & 59.0 & 70.5 & 86.0 & 2.7 \\


,avg_jaccard_cmupanoptic_3dgs,delta_x_avg_cmupanoptic_3dgs,occlusion_acc_cmupanoptic_3dgs,mte_cmupanoptic_3dgs,avg_jaccard_dex_ycb,delta_x_avg_dex_ycb,occlusion_acc_dex_ycb,mte_dex_ycb,avg_jaccard_kubric-multiview,delta_x_avg_kubric-multiview,occlusion_acc_kubric-multiview,mte_kubric-multiview
experiment,,,,,,,,,,,,
ablation_dataset_static_kub_only_f4wpvece_canonical_cached,78.046903,91.211239,88.706304,4.054027,81.982745,90.331537,92.725213,0.768206,64.057986,75.001804,87.693174,2.080717
main_pretrain_e25w9v86_canonical_cached,75.566621,91.675912,86.138679,3.932857,80.793698,90.745596,91.147060,0.737608,58.975121,70.549932,85.993011,2.737450


In [60]:
# encoder size ablation
experiments = ['ablation_encoder_size_base_hv8nguwx_canonical_cached', 'main_pretrain_e25w9v86_canonical_cached', 'ablation_encoder_size_giant_ubk4lvez_canonical_cached', 'ablation_encoder_size_gigantic_u1c34ou7_canonical_cached']
datasets = ['kubric-multiview']
metrics = ['avg_jaccard', 'delta_x_avg', 'occlusion_acc', 'mte']

cols = [f'{metric}_{dataset}' for dataset in datasets for metric in metrics]
sub_df = df.loc[experiments, cols]
df_to_latex(sub_df, decimals = 1, include_header = True)
sub_df

Index(['ablation_encoder_size_base_hv8nguwx_canonical_cached',
       'main_pretrain_e25w9v86_canonical_cached',
       'ablation_encoder_size_giant_ubk4lvez_canonical_cached',
       'ablation_encoder_size_gigantic_u1c34ou7_canonical_cached'],
      dtype='object', name='experiment')
avg_jaccard_kubric-multiview & delta_x_avg_kubric-multiview & occlusion_acc_kubric-multiview & mte_kubric-multiview \\
55.3 & 66.6 & 84.9 & 3.4 \\
59.0 & 70.5 & 86.0 & 2.7 \\
60.7 & 71.9 & 86.9 & 2.5 \\
61.0 & 72.7 & 86.5 & 2.5 \\


,avg_jaccard_kubric-multiview,delta_x_avg_kubric-multiview,occlusion_acc_kubric-multiview,mte_kubric-multiview
experiment,,,,
ablation_encoder_size_base_hv8nguwx_canonical_cached,55.264623,66.564418,84.916448,3.363138
main_pretrain_e25w9v86_canonical_cached,58.975121,70.549932,85.993011,2.737450
ablation_encoder_size_giant_ubk4lvez_canonical_cached,60.655728,71.940009,86.903138,2.510178
ablation_encoder_size_gigantic_u1c34ou7_canonical_cached,60.996526,72.661015,86.511864,2.510920


In [50]:
# latent dim ablation
# experiments = ['hxgceji7_ablation_latent_dim_256', '2p9of8sk_main_pretrain', 'c8a16qaw_ablation_latent_dim_1024']
experiments = ['ablation_latent_dim_256_lczwsdx3_canonical_cached', 'main_pretrain_e25w9v86_canonical_cached', 'ablation_latent_dim_1024_wk2i4o1u_canonical_cached']
datasets = ['kubric-multiview']
metrics = ['avg_jaccard', 'delta_x_avg', 'occlusion_acc', 'mte']

cols = [f'{metric}_{dataset}' for dataset in datasets for metric in metrics]
sub_df = df.loc[experiments, cols]
df_to_latex(sub_df, decimals = 1, include_header = True)
sub_df

Index(['ablation_latent_dim_256_lczwsdx3_canonical_cached',
       'main_pretrain_e25w9v86_canonical_cached',
       'ablation_latent_dim_1024_wk2i4o1u_canonical_cached'],
      dtype='object', name='experiment')
avg_jaccard_kubric-multiview & delta_x_avg_kubric-multiview & occlusion_acc_kubric-multiview & mte_kubric-multiview \\
55.6 & 66.7 & 85.3 & 3.1 \\
59.0 & 70.5 & 86.0 & 2.7 \\
58.5 & 70.3 & 85.8 & 2.8 \\


,avg_jaccard_kubric-multiview,delta_x_avg_kubric-multiview,occlusion_acc_kubric-multiview,mte_kubric-multiview
experiment,,,,
ablation_latent_dim_256_lczwsdx3_canonical_cached,55.642417,66.702225,85.320452,3.121618
main_pretrain_e25w9v86_canonical_cached,58.975121,70.549932,85.993011,2.737450
ablation_latent_dim_1024_wk2i4o1u_canonical_cached,58.547358,70.305567,85.752038,2.815537


### tapvid3d

In [ ]:
# # comparison to d4rt on tapvid3d
# experiments = ['main_finetune_vzkjqrys_final']
# datasets = ['tapvid3d_drivetrack', 'tapvid3d_adt', 'tapvid3d_pstudio']
# metrics = ['avg_jaccard_cam', 'occlusion_acc_cam'] #TODO: check ap3d/L1 metric they use

# cols = [f'{metric}_{dataset}' for dataset in datasets for metric in metrics]
# sub_df = df.loc[experiments, cols]

# # print('cotracker')
# # df_to_latex(cotracker_df[cols], decimals = 1, include_header = True)

# print('tracktail')
# df_to_latex(sub_df, decimals = 1, include_header = True)
# sub_df

KeyError: "['occlusion_acc_cam_tapvid3d_drivetrack', 'occlusion_acc_cam_tapvid3d_adt', 'occlusion_acc_cam_tapvid3d_pstudio'] not in index"

In [54]:
exp = ['main_finetune_vzkjqrys_final']
ds_cam   = ['tapvid3d_drivetrack', 'tapvid3d_adt', 'tapvid3d_pstudio']   # D4RT column order
ds_world = ['tapvid3d_drivetrack', 'tapvid3d_adt']

cam_cols   = [f'{m}_{d}' for d in ds_cam   for m in ['avg_jaccard_cam', 'delta_x_avg_cam', 'occlusion_acc']]
world_cols = [f'{m}_{d}' for d in ds_world for m in ['delta_x_avg_world', 'l1_world']]

df_to_latex(df.loc[exp, cam_cols],   decimals = 3, include_header = True)
df_to_latex(df.loc[exp, world_cols], decimals = 3, include_header = True)


Index(['main_finetune_vzkjqrys_final'], dtype='object', name='experiment')
avg_jaccard_cam_tapvid3d_drivetrack & delta_x_avg_cam_tapvid3d_drivetrack & occlusion_acc_tapvid3d_drivetrack & avg_jaccard_cam_tapvid3d_adt & delta_x_avg_cam_tapvid3d_adt & occlusion_acc_tapvid3d_adt & avg_jaccard_cam_tapvid3d_pstudio & delta_x_avg_cam_tapvid3d_pstudio & occlusion_acc_tapvid3d_pstudio \\
0.246 & 0.499 & 0.423 & 0.129 & 0.456 & 0.202 & 0.427 & 0.725 & 0.628 \\
Index(['main_finetune_vzkjqrys_final'], dtype='object', name='experiment')
delta_x_avg_world_tapvid3d_drivetrack & l1_world_tapvid3d_drivetrack & delta_x_avg_world_tapvid3d_adt & l1_world_tapvid3d_adt \\
0.510 & 0.343 & 0.459 & 1.374 \\


In [ ]:
# # comparison to mvtracker on tapvid2d
# # TODO: add thresholded aj and dx? to metrics
# experiments = ['TODO_main_finetune']
# datasets = ['tapvid2d']
# metrics = ['avg_jaccard', 'delta_x_avg', 'occlusion_acc', 'mte']

# cols = [f'{metric}_{dataset}' for dataset in datasets for metric in metrics]
# sub_df = df.loc[experiments, cols]
# df_to_latex(sub_df, decimals = 1, include_header = True)
# sub_df

In [14]:
# cotracker failed triangulations: share of pairs failed, and the split of those failures into the two reasons
stats = pd.read_csv('/home/ruppk2@hhmi.org/tracktail-paper/data/v2/cotracker_triangulation_stats.csv')

# one label per run, kept in the index (df_to_latex prints the index first, then the rows of numbers)
stats['run'] = stats['dataset'] + ' (' + stats['cameras'].str.replace(' (canonical)', '', regex=False) + ' cams)'
stats = stats.set_index('run')

table = stats[['pct_pairs_failed',
               'pct_of_failures_not_visible_enough',
               'pct_of_failures_visible_enough']]
table.columns = ['Failed', 'Not visible enough', 'Visible enough']

df_to_latex(table, decimals=1)


Index(['kubric-multiview (4 cams)', 'dex_ycb (4 cams)',
       'cmupanoptic_3dgs (4 cams)', 'point-odyssey-human (4 cams)',
       'point-odyssey-human (9 cams)', 'po-animal-sub256-motion (4 cams)',
       'po-animal-sub256-motion (9 cams)'],
      dtype='object', name='run')
Failed & Not visible enough & Visible enough \\
40.9 & 60.5 & 39.5 \\
10.6 & 64.4 & 35.6 \\
20.2 & 51.4 & 48.6 \\
48.7 & 26.7 & 73.3 \\
29.6 & 7.3 & 92.7 \\
52.0 & 28.8 & 71.2 \\
30.4 & 4.6 & 95.4 \\
